# Insurance Charges Analysis — Data Science Project

**Dataset:** Medical Insurance Costs — 1,338 records, 7 features  
**Objective:** Predict insurance charges · Classify high-cost customers · Segment customers into risk profiles

---

## Table of Contents
1. [Setup & Data Loading](#setup)
2. [Part I — Exploratory Data Analysis](#eda)
3. [Part II — Regression Models](#regression)
4. [Part III — Binary Risk Classification](#classification)
5. [Part IV — Customer Risk Segmentation *(Innovative)*](#clustering)


---
<a id='setup'></a>
## 0. Setup — Imports & Dataset

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Supervised learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

# Unsupervised learning
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries loaded ✓')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('insurance.csv')
print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
display(df.head())
display(df.describe())

In [ ]:
# ── Encode categoricals (used throughout) ────────────────────────────────────
df_enc = df.copy()
df_enc['sex']    = (df_enc['sex']    == 'male').astype(int)
df_enc['smoker'] = (df_enc['smoker'] == 'yes').astype(int)
df_enc['region'] = LabelEncoder().fit_transform(df_enc['region'])
print('Encoding done. Columns:', df_enc.columns.tolist())

---
<a id='eda'></a>
## Part I — Exploratory Data Analysis

### 1.1  Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Figure 1 – Feature Distributions', fontsize=14, fontweight='bold')

for i, col in enumerate(['age', 'bmi', 'children', 'charges']):
    ax = axes[i // 3][i % 3]
    ax.hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(col.capitalize()); ax.set_xlabel(col); ax.set_ylabel('Count')

df['smoker'].value_counts().plot(
    kind='pie', ax=axes[1][1], autopct='%1.1f%%',
    colors=['#4CAF50', '#F44336'], labels=['Non-smoker', 'Smoker'])
axes[1][1].set_title('Smoker proportion')

df['sex'].value_counts().plot(
    kind='bar', ax=axes[1][2],
    color=['#2196F3', '#E91E63'], edgecolor='white', rot=0)
axes[1][2].set_title('Sex distribution')

plt.tight_layout()
plt.show()

**Observations:**
- Age and BMI follow near-normal distributions
- Charges are strongly right-skewed (smoker-driven high-cost tail)
- ~20% of policyholders are smokers, consistent with the general US population

### 1.2  Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, mask=mask, linewidths=0.5)
ax.set_title('Figure 2 – Pearson Correlation Matrix',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with charges (sorted):')
print(corr['charges'].drop('charges').sort_values(ascending=False))

**Observations:**
- `smoker` has by far the highest correlation with charges (r ≈ 0.79)
- `age` (r ≈ 0.30) and `bmi` (r ≈ 0.20) are secondary predictors
- `sex` and `region` are essentially uncorrelated with charges

### 1.3  Smoking–Charges Interaction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 3 – Charges by Smoking Status',
             fontsize=13, fontweight='bold')

sns.boxplot(data=df, x='smoker', y='charges',
            palette={'yes': '#F44336', 'no': '#4CAF50'}, ax=axes[0])
axes[0].set_title('Charge distribution by smoker status')

sns.scatterplot(data=df, x='age', y='charges', hue='smoker',
                palette={'yes': '#F44336', 'no': '#4CAF50'},
                alpha=0.55, ax=axes[1])
axes[1].set_title('Age vs. Charges (coloured by smoker)')

plt.tight_layout()
plt.show()

print('Mean charges by smoking status:')
print(df.groupby('smoker')['charges'].mean().round(2))
print(f'\nSmoker premium: ${df[df.smoker=="yes"]["charges"].mean() - df[df.smoker=="no"]["charges"].mean():,.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(df['bmi'], df['charges'],
                c=(df['smoker'] == 'yes').astype(int),
                cmap='RdYlGn_r', alpha=0.5, edgecolors='none')
ax.set_title('Figure 4 – BMI vs. Charges by Smoking Status',
             fontsize=13, fontweight='bold')
ax.set_xlabel('BMI'); ax.set_ylabel('Charges ($)')
plt.colorbar(sc, label='Smoker (1 = yes)')
plt.tight_layout()
plt.show()

**Observations:**
- Smokers pay ~3× more than non-smokers on average
- The gap compounds with age and persists across all BMI levels
- Two visually distinct strata are clearly separable in the BMI–charges plane

---
<a id='regression'></a>
## Part II — Regression Models

### 2.1  Preprocessing & Train/Test Split

In [ ]:
X = df_enc.drop('charges', axis=1)
y = df_enc['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train only
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print('Standardisation applied (fit on train only to prevent leakage).')

### 2.2  Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    results[name] = {
        'R2':    round(r2_score(y_test, preds), 4),
        'MAE':   round(mean_absolute_error(y_test, preds), 2),
        'RMSE':  round(np.sqrt(mean_squared_error(y_test, preds)), 2),
        'preds': preds
    }

results_df = pd.DataFrame({
    k: {m: v for m, v in d.items() if m != 'preds'}
    for k, d in results.items()
}).T
display(results_df)

In [ ]:
# Figure 5 – Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Figure 5 – Regression Model Comparison',
             fontsize=13, fontweight='bold')
cols = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

for i, metric in enumerate(['R2', 'MAE', 'RMSE']):
    vals = [results[m][metric] for m in results]
    bars = axes[i].bar(list(results.keys()), vals, color=cols, edgecolor='white')
    axes[i].set_title(metric)
    axes[i].set_xticklabels(list(results.keys()), rotation=15, ha='right')
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() * 1.01, str(val),
                     ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

### 2.3  Random Forest Deep Dive

In [ ]:
rf    = models['Random Forest']
preds = results['Random Forest']['preds']

# Figure 6 – Actual vs. Predicted + Residuals
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 6 – Random Forest: Actual vs. Predicted',
             fontsize=13, fontweight='bold')

axes[0].scatter(y_test, preds, alpha=0.4, color='steelblue', edgecolors='none')
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5)
axes[0].set_xlabel('Actual ($)'); axes[0].set_ylabel('Predicted ($)')
axes[0].set_title('Actual vs. Predicted')

residuals = y_test - preds
axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', ls='--')
axes[1].set_xlabel('Residuals ($)'); axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 7 – Feature Importances
feat_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance')

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feat_df['feature'], feat_df['importance'],
        color='steelblue', edgecolor='white')
ax.set_title('Figure 7 – Random Forest Feature Importances',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### 2.4  Cross-Validation Stability

In [ ]:
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(
        model, scaler.fit_transform(X), y, cv=5, scoring='r2')
    cv_results[name] = scores
    print(f'{name:22s}  mean={scores.mean():.4f}  std={scores.std():.4f}')

# Figure 8 – CV boxplot
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(cv_results.values(), labels=cv_results.keys(),
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', color='navy'),
           medianprops=dict(color='red', linewidth=2))
ax.set_title('Figure 8 – 5-Fold Cross-Validation R² Scores',
             fontsize=13, fontweight='bold')
ax.set_ylabel('R² Score')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

**Observations:**
- All models generalise well — low inter-fold variance across all 5 folds
- Gradient Boosting shows the narrowest interquartile range (most consistent)
- No evidence of overfitting in any model

---
<a id='classification'></a>
## Part III — Binary Risk Classification

### 3.1  Problem Definition

In [ ]:
# Binary target: above/below median charge
df_class = df_enc.copy()
median_charge = df_class['charges'].median()
df_class['high_charges'] = (df_class['charges'] > median_charge).astype(int)

print(f'Median charge threshold: ${median_charge:,.2f}')
print('Class distribution:')
print(df_class['high_charges'].value_counts().rename({0:'Low Charges', 1:'High Charges'}))

### 3.2  Training & Evaluation

In [ ]:
X_c = df_class.drop(['charges', 'high_charges'], axis=1)
y_c = df_class['high_charges']

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42)

Xc_sc  = scaler.fit_transform(X_c_train)
Xct_sc = scaler.transform(X_c_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(Xc_sc, y_c_train)
y_pred_c = clf.predict(Xct_sc)

print('Classification Report:')
print(classification_report(y_c_test, y_pred_c,
      target_names=['Low Charges', 'High Charges']))

In [ ]:
# Figure 9 – Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 9 – Classification: High vs. Low Charges',
             fontsize=13, fontweight='bold')

ConfusionMatrixDisplay(
    confusion_matrix(y_c_test, y_pred_c),
    display_labels=['Low', 'High']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

y_prob = clf.predict_proba(Xct_sc)[:, 1]
fpr, tpr, _ = roc_curve(y_c_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], 'r--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'AUC-ROC: {roc_auc:.4f}')

**Observations:**
- AUC-ROC = 0.98 — near-perfect discriminative power
- Precision and Recall both reach 0.98 for both classes
- The classifier is production-ready for binary risk segmentation

---
<a id='clustering'></a>
## Part IV — Customer Risk Segmentation *(Innovative Contribution)*

> **Motivation:** Supervised models predict individual charges but do not reveal *customer archetypes*.
> K-Means clustering groups policyholders into interpretable risk profiles, enabling targeted
> underwriting, personalised premiums, and wellness programme design.

### 4.1  Feature Matrix Preparation

In [ ]:
# Include all features (charges included) for actuarially meaningful segments
X_all = df_enc.values
scaler_c = StandardScaler()
X_sc = scaler_c.fit_transform(X_all)

print('Feature matrix shape:', X_sc.shape)
print('Features:', df_enc.columns.tolist())

### 4.2  Choosing the Optimal Number of Clusters

In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sc)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sc, labels))

# Figure 10 – Elbow + Silhouette
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Figure 10 – Optimal Number of Clusters',
             fontsize=13, fontweight='bold')

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', lw=2)
axes[0].axvline(3, color='red', ls='--', alpha=0.7, label='k = 3 selected')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method'); axes[0].legend()

axes[1].plot(list(K_range), silhouettes, 's-', color='#FF9800', lw=2)
axes[1].axvline(3, color='red', ls='--', alpha=0.7, label='k = 3 selected')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score'); axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Silhouette at k=3: {silhouettes[1]:.4f}')

**Observations:**
- The elbow bends sharply at k = 3
- The silhouette score is locally optimal at k = 3
- k = 3 aligns with intuitive actuarial categories: Low / Medium / High risk

### 4.3  Applying K-Means and Labelling Segments

In [ ]:
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df_enc['cluster'] = km_final.fit_predict(X_sc)
df['cluster']     = df_enc['cluster']

# Relabel clusters by ascending mean charge: 0=Low, 1=Medium, 2=High
order = df.groupby('cluster')['charges'].mean().sort_values().index.tolist()
remap = {order[0]: 0, order[1]: 1, order[2]: 2}
df['cluster']     = df['cluster'].map(remap)
df_enc['cluster'] = df_enc['cluster'].map(remap)

PALETTE   = {0: '#4CAF50', 1: '#FF9800', 2: '#F44336'}
LABEL_MAP = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}

print('Cluster sizes:')
print(df['cluster'].value_counts().sort_index().rename(LABEL_MAP))
print(f'\nFinal silhouette score: {silhouette_score(X_sc, df_enc["cluster"]):.4f}')

### 4.4  Visualisation — PCA Projection & Charge Distribution

In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)
v1, v2 = pca.explained_variance_ratio_ * 100

# Figure 11 – PCA scatter + boxplot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 11 – Customer Risk Segments (K-Means, k = 3)',
             fontsize=13, fontweight='bold')

for c, name in LABEL_MAP.items():
    mask = df['cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=PALETTE[c], label=name, alpha=0.55, edgecolors='none', s=25)
axes[0].set_xlabel(f'PC1 ({v1:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({v2:.1f}% variance)')
axes[0].set_title('PCA 2D Projection')
axes[0].legend(fontsize=9)

bp = axes[1].boxplot(
    [df[df['cluster'] == c]['charges'].values for c in [0, 1, 2]],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, c in zip(bp['boxes'], [0, 1, 2]):
    patch.set_facecolor(PALETTE[c])
axes[1].set_xticklabels(['Low Risk', 'Medium Risk', 'High Risk'])
axes[1].set_ylabel('Annual Charges ($)')
axes[1].set_title('Charge Distribution per Segment')

plt.tight_layout()
plt.show()

### 4.5  Risk Profile Analysis — Radar Chart

In [ ]:
profile_cols = ['age', 'bmi', 'children', 'smoker', 'charges']
profile = df_enc.groupby('cluster')[profile_cols].mean()
profile.index = ['Low Risk', 'Medium Risk', 'High Risk']

# Normalise 0–1 for radar display
profile_norm = (
    (profile - profile.min()) / (profile.max() - profile.min())
)

# Figure 12 – Radar chart
categories = ['Age', 'BMI', 'Children', 'Smoker %', 'Charges']
N      = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_title('Figure 12 – Cluster Profiles (Radar Chart)',
             fontsize=13, fontweight='bold', pad=20)

for idx, (label, color) in enumerate(
        zip(profile_norm.index, [PALETTE[0], PALETTE[1], PALETTE[2]])):
    values = profile_norm.loc[label].tolist() + [profile_norm.loc[label].tolist()[0]]
    ax.plot(angles, values, 'o-', lw=2.5, color=color, label=label)
    ax.fill(angles, values, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_yticklabels([])
ax.legend(loc='upper right', bbox_to_anchor=(1.42, 1.15), fontsize=10)
plt.tight_layout()
plt.show()

### 4.6  Business Summary

In [ ]:
print('=' * 65)
print('  CUSTOMER RISK SEGMENTATION — ACTUARIAL SUMMARY')
print('=' * 65)

for c in [0, 1, 2]:
    seg       = df[df['cluster'] == c]
    smoke_pct = df_enc[df_enc['cluster'] == c]['smoker'].mean() * 100
    print(f'\n  {LABEL_MAP[c].upper()}  (n={len(seg)}, {len(seg)/len(df)*100:.0f}% of portfolio)')
    print(f'    Smoker rate     : {smoke_pct:.0f}%')
    print(f'    Mean charges    : ${seg["charges"].mean():>10,.2f}')
    print(f'    Median charges  : ${seg["charges"].median():>10,.2f}')
    print(f'    Mean age        : {seg["age"].mean():>9.1f} yrs')
    print(f'    Mean BMI        : {seg["bmi"].mean():>9.1f}')

total_charges = df['charges'].sum()
hr_charges    = df[df['cluster'] == 2]['charges'].sum()
print(f'\n  High Risk segment = {len(df[df["cluster"]==2])/len(df)*100:.0f}% of customers')
print(f'  but represents   = {hr_charges/total_charges*100:.0f}% of total portfolio charges')

print('\n' + '=' * 65)
print('  BUSINESS RECOMMENDATIONS')
print('=' * 65)
recs = [
    'Apply +200% loading factor to High Risk premium plans',
    'Target smoking cessation programmes exclusively at High Risk',
    'Standard pricing for Low Risk; marginal adjustment for Medium Risk',
    'Re-run segmentation annually as policyholder composition shifts',
    'Flag outliers (large distance to cluster centroid) for manual review',
]
for i, r in enumerate(recs, 1):
    print(f'  {i}. {r}')

---

## Summary of Results

| Task | Best Model | Key Metric |
|---|---|---|
| Regression | Gradient Boosting | R² = 0.9635, RMSE ≈ $1,935 |
| Classification | Random Forest | AUC-ROC = 0.98 |
| Segmentation | K-Means (k=3) | Silhouette = 0.23 |

**Core finding:** Smoking status drives charges more than any other variable — across regression importance, correlation, and unsupervised clustering. The High Risk segment (20% of customers, 100% smokers) accounts for ~45% of total portfolio charges.

---

## Difficulties Encountered

- **Signal dominance:** The `smoker` variable is so strong that Linear Regression competes with ensemble methods — atypical for tabular data and worth noting methodologically
- **Scaler leakage:** StandardScaler must be fit on train data only; applying it to the full dataset before splitting would inflate test performance
- **K-Means on mixed types:** Euclidean distance is suboptimal for binary features (smoker, sex); Gower distance or Gaussian Mixture Models are more principled alternatives
- **Cluster relabelling:** Cluster indices from K-Means are arbitrary; domain knowledge (sort by mean charge) is required to assign meaningful labels

## Perspectives for Improvement

- **SHAP values:** True Shapley-based per-policyholder feature attribution for regulatory transparency
- **Gaussian Mixture Models:** Soft cluster memberships reflecting probabilistic risk
- **DBSCAN:** Detect anomalous policyholders who do not fit any standard risk profile
- **Richer features:** Claims history, pre-existing conditions, medication use, ZIP code
- **Interactive premium calculator:** Feed cluster assignment + regression prediction into a real-time web tool